# Final Stage F: Nine-Mask Forensic Extractor

**Runtime:** CPU.

This diagnostic uses the two regenerated 2,718-call capture manifests to identify the nine calls whose Boolean mask hashes differ between Notebook 51R2 and Notebook 53R2. It then loads only those 18 persisted `mask.npy` files and exports the actual XOR coordinates to ordinary CSV files for review.

It does **not** rerun reconstruction, modify Notebook 51R2/53R2 outputs, or scan every captured NumPy array.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configuration

Edit only the two regenerated capture-root paths below if necessary. Each capture root is expected to contain `captured_calls.csv` and `calls/call_XXXXXX/mask.npy`.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# ------------------------------------------------------------------
# EDIT THESE TWO PATHS
# ------------------------------------------------------------------
CAPTURE_51 = ROOT / 'REPLACE_WITH_REGENERATED_51R2_CAPTURE_ROOT'
CAPTURE_53 = ROOT / 'REPLACE_WITH_REGENERATED_53R2_CAPTURE_ROOT'

MANIFEST_51 = CAPTURE_51 / 'captured_calls.csv'
MANIFEST_53 = CAPTURE_53 / 'captured_calls.csv'

OUT = ROOT / 'stage_f_nine_mask_extract'
OUT.mkdir(parents=True, exist_ok=True)

print('51R2 capture:', CAPTURE_51)
print('53R2 capture:', CAPTURE_53)
print('Output:', OUT)


## 2. Helpers and regenerated-manifest integrity

This gate requires a complete, duplicate-free `1..2718` call population on both sides before any mask array is loaded.


In [ ]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def bool_hash(array):
    value = np.ascontiguousarray(np.asarray(array) > 0)
    h = hashlib.sha256()
    h.update(str(value.dtype).encode())
    h.update(str(value.shape).encode())
    h.update(value.tobytes())
    return h.hexdigest()


def choose(frame, names, label):
    for name in names:
        if name in frame.columns:
            return name
    raise RuntimeError(
        f'Could not resolve {label}. Columns: {frame.columns.tolist()}'
    )


def load_manifest(path, side):
    require(path.exists(), f'{side}: manifest not found: {path}')
    frame = pd.read_csv(path, low_memory=False)

    require(
        'call_number' in frame.columns,
        f'{side}: no call_number column. Columns: {frame.columns.tolist()}'
    )

    frame['call_number'] = pd.to_numeric(
        frame['call_number'], errors='raise'
    ).astype(int)

    require(
        not frame['call_number'].duplicated().any(),
        f'{side}: duplicate call_number values.'
    )

    require(
        len(frame) == 2718
        and frame['call_number'].min() == 1
        and frame['call_number'].max() == 2718
        and frame['call_number'].nunique() == 2718,
        f'{side}: expected complete calls 1..2718; '
        f'rows={len(frame)}, unique={frame["call_number"].nunique()}, '
        f'range={frame["call_number"].min()}..{frame["call_number"].max()}'
    )

    return frame.sort_values('call_number').reset_index(drop=True)


m51 = load_manifest(MANIFEST_51, '51R2')
m53 = load_manifest(MANIFEST_53, '53R2')

print('51R2 calls:', len(m51))
print('53R2 calls:', len(m53))


## 3. Identify the nine regenerated mask-hash differences

The comparison uses the semantic Boolean-mask hash from each regenerated manifest. If the current regenerated evidence does not contain exactly nine differences, the notebook stops and writes the unexpected comparison for inspection.


In [ ]:
mask_col_51 = choose(
    m51,
    ['mask_boolean_sha256', 'mask_boolean_hash', 'mask_sha256', 'mask_hash'],
    '51R2 mask hash'
)
mask_col_53 = choose(
    m53,
    ['mask_boolean_sha256', 'mask_boolean_hash', 'mask_sha256', 'mask_hash'],
    '53R2 mask hash'
)

joined = m51.merge(
    m53,
    on='call_number',
    suffixes=('_51', '_53'),
    how='outer',
    indicator=True,
    validate='one_to_one'
)

require(
    (joined['_merge'] == 'both').all(),
    'The regenerated call populations do not align one-to-one by absolute call number.'
)

diff = joined.loc[
    joined[f'{mask_col_51}_51'].astype(str)
    != joined[f'{mask_col_53}_53'].astype(str)
].copy()

if len(diff) != 9:
    path = OUT / 'unexpected_mask_hash_differences.csv'
    diff.to_csv(path, index=False)
    raise RuntimeError(
        f'Expected exactly 9 regenerated mask-hash differences, found {len(diff)}. '
        f'See {path}'
    )

print('Differing calls:', len(diff))
print(diff['call_number'].astype(int).tolist())


## 4. Resolve only the 18 mask files

No recursive capture-directory indexing is performed. The paths are constructed directly from the nine absolute call numbers established above.


In [ ]:
resolved_rows = []

for _, row in diff.sort_values('call_number').iterrows():
    call = int(row['call_number'])

    for side, root in [('51R2', CAPTURE_51), ('53R2', CAPTURE_53)]:
        path = root / 'calls' / f'call_{call:06d}' / 'mask.npy'
        resolved_rows.append({
            'side': side,
            'call_number': call,
            'mask_path': str(path),
            'exists': path.exists(),
        })

resolved = pd.DataFrame(resolved_rows)
resolved_path = OUT / 'resolved_18_mask_paths.csv'
resolved.to_csv(resolved_path, index=False)

display(resolved)

missing = resolved.loc[~resolved['exists']]
require(
    missing.empty,
    f'At least one of the 18 mask files is missing. See {resolved_path}'
)


## 5. Compare the nine actual mask pairs

For each pair the notebook calculates `SHARED`, `ONLY_51R2`, `ONLY_53R2`, and `XOR`, verifies each persisted `.npy` against its regenerated manifest hash, and writes one row for every differing coordinate.


In [ ]:
page_candidates_51 = [
    c for c in ['page_51', 'context_page_51', 'page_name_51']
    if c in diff.columns
]
page_candidates_53 = [
    c for c in ['page_53', 'context_page_53', 'page_name_53']
    if c in diff.columns
]
page_col_51 = page_candidates_51[0] if page_candidates_51 else None
page_col_53 = page_candidates_53[0] if page_candidates_53 else None

summary_rows = []
coordinate_rows = []

for _, row in diff.sort_values('call_number').iterrows():
    call = int(row['call_number'])

    path51 = CAPTURE_51 / 'calls' / f'call_{call:06d}' / 'mask.npy'
    path53 = CAPTURE_53 / 'calls' / f'call_{call:06d}' / 'mask.npy'

    array51 = np.load(path51, allow_pickle=False)
    array53 = np.load(path53, allow_pickle=False)

    require(
        array51.shape == array53.shape,
        f'Call {call}: mask shape mismatch {array51.shape} vs {array53.shape}'
    )

    mask51 = array51 > 0
    mask53 = array53 > 0

    only51 = mask51 & ~mask53
    only53 = mask53 & ~mask51
    shared = mask51 & mask53
    xor = mask51 ^ mask53

    page51 = str(row[page_col_51]).strip() if page_col_51 else ''
    page53 = str(row[page_col_53]).strip() if page_col_53 else ''
    require(
        not (page51 and page53 and page51 != page53),
        f'Call {call}: page mismatch {page51} vs {page53}'
    )
    page = page51 or page53

    computed51 = bool_hash(array51)
    computed53 = bool_hash(array53)
    expected51 = str(row[f'{mask_col_51}_51'])
    expected53 = str(row[f'{mask_col_53}_53'])

    require(
        computed51 == expected51,
        f'Call {call}: persisted 51R2 mask does not match regenerated manifest.'
    )
    require(
        computed53 == expected53,
        f'Call {call}: persisted 53R2 mask does not match regenerated manifest.'
    )

    summary_rows.append({
        'call_number': call,
        'page': page,
        'height': int(array51.shape[0]),
        'width': int(array51.shape[1]),
        'mask_pixels_51R2': int(mask51.sum()),
        'mask_pixels_53R2': int(mask53.sum()),
        'shared_pixels': int(shared.sum()),
        'only_51R2_pixels': int(only51.sum()),
        'only_53R2_pixels': int(only53.sum()),
        'xor_pixels': int(xor.sum()),
        'boolean_sha256_51R2': computed51,
        'boolean_sha256_53R2': computed53,
        '51_hash_matches_manifest': True,
        '53_hash_matches_manifest': True,
    })

    ys, xs = np.where(xor)
    for y, x in zip(ys.tolist(), xs.tolist()):
        coordinate_rows.append({
            'call_number': call,
            'page': page,
            'y': int(y),
            'x': int(x),
            'in_51R2_mask': bool(mask51[y, x]),
            'in_53R2_mask': bool(mask53[y, x]),
            'difference_side': 'ONLY_51R2' if mask51[y, x] else 'ONLY_53R2',
            'mask_value_51R2': int(array51[y, x]),
            'mask_value_53R2': int(array53[y, x]),
        })

    del array51, array53, mask51, mask53, only51, only53, shared, xor

summary = pd.DataFrame(summary_rows).sort_values(['page', 'call_number']).reset_index(drop=True)
coordinates = pd.DataFrame(coordinate_rows).sort_values(
    ['page', 'call_number', 'y', 'x']
).reset_index(drop=True)

summary_path = OUT / 'nine_mask_pair_summary.csv'
coordinate_path = OUT / 'nine_mask_coordinate_differences.csv'

summary.to_csv(summary_path, index=False)
coordinates.to_csv(coordinate_path, index=False)

display(summary)
print('Coordinate rows:', len(coordinates))


In [ ]:
# temporary diagnostic cell

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd

CALL = 108

m51 = pd.read_csv(MANIFEST_51)
row51 = m51.loc[
    pd.to_numeric(
        m51['call_number'],
        errors='raise'
    ).astype(int) == CALL
].iloc[0]

path51 = (
    CAPTURE_51
    / 'calls'
    / f'call_{CALL:06d}'
    / 'mask.npy'
)

a = np.load(
    path51,
    allow_pickle=False
)

expected = str(
    row51['mask_boolean_sha256']
)

def semantic_hash(x):
    x = np.ascontiguousarray(x)

    h = hashlib.sha256()
    h.update(str(x.dtype).encode())
    h.update(str(x.shape).encode())
    h.update(x.tobytes())

    return h.hexdigest()

variants = {

    'original':
        semantic_hash(a),

    'bool':
        semantic_hash(
            a > 0
        ),

    'uint8_01':
        semantic_hash(
            (a > 0).astype(np.uint8)
        ),

    'uint8_255':
        semantic_hash(
            (a > 0).astype(np.uint8) * 255
        ),
}

print('Call:', CALL)
print('Path:', path51)
print('Shape:', a.shape)
print('dtype:', a.dtype)
print('unique values:', np.unique(a))
print()
print('Manifest mask_boolean_sha256:')
print(expected)

print('\nCandidate hashes:')

for name, value in variants.items():

    print(
        name.ljust(12),
        value,
        '<-- MATCH'
        if value == expected
        else ''
    )

## 6. Final forensic summary

These are the two CSVs to upload for review. The coordinate CSV is the key output because it expresses the inaccessible NumPy masks as ordinary `(page, call, y, x)` evidence.


In [ ]:
verdict = {
    'regenerated_calls_per_side': 2718,
    'differing_mask_calls': int(len(summary)),
    'total_xor_pixels': int(summary['xor_pixels'].sum()),
    'total_only_51R2_pixels': int(summary['only_51R2_pixels'].sum()),
    'total_only_53R2_pixels': int(summary['only_53R2_pixels'].sum()),
    '53R2_masks_are_strict_supersets_for_all_nine': bool(
        (summary['only_51R2_pixels'] == 0).all()
        and (summary['only_53R2_pixels'] > 0).all()
    ),
    '51R2_masks_are_strict_supersets_for_all_nine': bool(
        (summary['only_53R2_pixels'] == 0).all()
        and (summary['only_51R2_pixels'] > 0).all()
    ),
    'all_persisted_masks_match_regenerated_manifests': bool(
        summary['51_hash_matches_manifest'].all()
        and summary['53_hash_matches_manifest'].all()
    ),
}

verdict_path = OUT / 'nine_mask_forensic_verdict.json'
verdict_path.write_text(json.dumps(verdict, indent=2))

print(json.dumps(verdict, indent=2))
print('
UPLOAD THESE TWO FILES:')
print(summary_path)
print(coordinate_path)
print('
Optional third file:')
print(verdict_path)
